### Langkah 1: Memuat dan Merapikan Data Asli (German Credit Dataset)

1. Membaca dataset asli data/german.data yang belum memiliki nama kolom.
2. Memberi nama kolom yang jelas dan representatif berdasarkan dokumentasi data/tutorial.md.
3. Menyelaraskan target label risk:
   - 1 $\rightarrow$ 1 (Layak / Good Credit)
   - 2 $\rightarrow$ 0 (Tidak Layak / Bad Credit)
4. Mengonversi nominal pinjaman (credit_amount) ke Rupiah (IDR) dengan skala proporsional realistis.
5. Menyimpan data bersih ini ke dalam file data/german_credit_clean_idr.csv agar siap digunakan pada tahap pemodelan.

### 1. Import Library & Penentuan Path

In [1]:
import pandas as pd
import os

# Pastikan path folder data tersedia
DATA_PATH = os.path.join("data", "german.data")
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"File tidak ditemukan di {DATA_PATH}")
print(f"File data ditemukan: {DATA_PATH}")

File data ditemukan: data\german.data


### 2. Tentukan Nama Kolom Berdasarkan data/tutorial.md
File german.data memiliki 20 atribut (fitur) dan 1 kolom target di posisi terakhir.

In [2]:
column_names = [
    "checking_status",       # Status rekening giro yang ada
    "duration_month",        # Durasi pinjaman dalam bulan
    "credit_history",        # Riwayat pembayaran kredit masa lalu
    "purpose",               # Tujuan penggunaan pinjaman (mobil, radio/tv, dsb)
    "credit_amount",         # Jumlah pinjaman (dalam DM)
    "savings_status",        # Saldo tabungan/obligasi
    "employment_length",     # Lama bekerja di tempat saat ini
    "installment_rate",      # Persentase cicilan terhadap disposable income
    "personal_status_sex",   # Status pernikahan dan jenis kelamin
    "other_debtors",         # Penjamin / debitur bersama
    "residence_since",       # Lama tinggal di tempat tinggal sekarang (tahun)
    "property",              # Kepemilikan aset/agunan (properti, asuransi, dll)
    "age",                   # Umur nasabah dalam tahun
    "other_installment_plans",# Cicilan lain di luar bank ini
    "housing",               # Status tempat tinggal (sewa, milik sendiri, gratis)
    "existing_credits",      # Jumlah kredit aktif di bank ini
    "job",                   # Kategori pekerjaan/keterampilan
    "num_dependents",        # Jumlah tanggungan
    "telephone",             # Kepemilikan telepon terdaftar
    "foreign_worker",        # Apakah pekerja asing
    "risk"                   # Target: 1 = Good (Layak), 2 = Bad (Tidak Layak)
]

# Membaca file german.data (pemisah antar kolom adalah spasi)
df = pd.read_csv(DATA_PATH, sep=' ', header=None, names=column_names)
print(f"Ukuran dataset: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head(1)

Ukuran dataset: 1000 baris, 21 kolom


,checking_status,duration_month,credit_history,purpose,credit_amount,savings_status,employment_length,installment_rate,personal_status_sex,other_debtors,...,property,age,other_installment_plans,housing,existing_credits,job,num_dependents,telephone,foreign_worker,risk
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1


### 3. Normalisasi Target Label (risk)
Di dataset asli:
- 1 = Nasabah Baik (Good)
- 2 = Nasabah Berisiko Gagal Bayar (Bad)

Dalam standar Machine Learning biner, kita petakan menjadi:
- 1 = Layak (Good)
- 0 = Tidak Layak (Bad)

In [3]:
# Ubah: 1 -> 1 (Layak), 2 -> 0 (Tidak Layak)
df['risk'] = df['risk'].map({1: 1, 2: 0})

# Cek distribusi target
print("Distribusi Label Risk:")
print(df['risk'].value_counts())
print("\nPersentase:")
print(df['risk'].value_counts(normalize=True) * 100)

Distribusi Label Risk:
risk
1    700
0    300
Name: count, dtype: int64

Persentase:
risk
1    70.0
0    30.0
Name: proportion, dtype: float64


### 4. Konversi Nominal Pinjaman ke Versi Rupiah (IDR)
Berdasarkan kesepakatan Opsi 1 (skala proporsional realistis):
- Kita kalikan nominal asli dengan 10.000 agar menghasilkan pinjaman dalam rentang wajar pinjaman perbankan/fintech Indonesia (sekitar Rp 2,5 juta hingga Rp 184 juta).
- Kita buat kolom baru credit_amount_idr.

In [4]:
SCALE_FACTOR = 10000
df['credit_amount_idr'] = df['credit_amount'] * SCALE_FACTOR

# Tampilkan ringkasan statistik nominal pinjaman dalam Rupiah
print("Ringkasan Pinjaman dalam Rupiah (IDR):")
print(f"Pinjaman Terkecil : Rp {df['credit_amount_idr'].min():,.0f}")
print(f"Pinjaman Terbesar : Rp {df['credit_amount_idr'].max():,.0f}")
print(f"Rata-rata Pinjaman: Rp {df['credit_amount_idr'].mean():,.0f}")

df[['credit_amount', 'credit_amount_idr', 'duration_month', 'age', 'risk']].head()

Ringkasan Pinjaman dalam Rupiah (IDR):
Pinjaman Terkecil : Rp 2,500,000
Pinjaman Terbesar : Rp 184,240,000
Rata-rata Pinjaman: Rp 32,712,580


,credit_amount,credit_amount_idr,duration_month,age,risk
0,1169,11690000,6,67,1
1,5951,59510000,48,22,0
2,2096,20960000,12,49,1
3,7882,78820000,42,45,1
4,4870,48700000,24,53,0


### 5. Cek Nilai Kosong (Missing Values)
Memastikan tidak ada data yang hilang/berlubang.

In [5]:
missing_values = df.isnull().sum()
print("Jumlah nilai kosong per kolom:")
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "Tidak ada nilai kosong (Data lengkap 100%)")

Jumlah nilai kosong per kolom:
Tidak ada nilai kosong (Data lengkap 100%)


### 6. Simpan Dataset Rapi & Bersih
Data diekspor ke file CSV baru data/german_credit_clean_idr.csv untuk dipakai di notebook pemodelan berikutnya.

In [6]:
OUTPUT_PATH = os.path.join("data", "german_credit_clean_idr.csv")
df.to_csv(OUTPUT_PATH, index=False)
print(f"Dataset berhasil disimpan di: {OUTPUT_PATH}")

Dataset berhasil disimpan di: data\german_credit_clean_idr.csv
